# Chapter 34
## Nested Gamma Theta Rhythms
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

A_CURRENT and PRE_OLM_X_INF_TAU_X are static plots of gating-variable
curves (no ODE simulation) -- Brian2 has nothing to add there. This
notebook covers the three single-neuron sub-examples (the network
sub-examples EIO_1/PING_WITH_THETA_DRIVE/PING_WITH_THETA_INHIBITION are
substantially larger 40-50-neuron networks and aren't covered here).

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np

### Figure 34.5
Pre-OLM Cell Voltage Trace

The pre-OLM cell is a WB-style Na/K neuron with C=1.3uF/cm^2 (not the
usual 1uF/cm^2 -- the book's own MATLAB script divides by `c` here,
unlike most other chapters where c=1 makes that division a no-op).

In [ ]:
def simulate_pre_OLM_neuron(i_ext, simulation_time, dt=0.01 * b2.ms):
    El = -70 * b2.mV
    EK = -100 * b2.mV
    ENa = 90 * b2.mV
    gl = 0.05 * b2.msiemens
    gK = 23 * b2.msiemens
    gNa = 30 * b2.msiemens
    C = 1.3 * b2.ufarad

    eqs = """
    I_e : amp

    alpham = (vm + 38*mV) / (10*mV) / (1.0 - exp(-(vm + 38*mV) / (10*mV))) /ms : Hz
    alphah = 0.07 * exp(-(vm + 63.0*mV) / (20.0*mV))/ms : Hz
    alphan = 0.018/mV * (vm - 25*mV) / (1.0 - exp(-(vm - 25*mV) / (25*mV)))/ms : Hz

    betam = 4.0 * exp(-(vm + 65.0*mV) / (18.0*mV))/ms : Hz
    betah = 1.0 / (exp(-(vm + 33.0*mV) / (10.0*mV)) + 1.0)/ms : Hz
    betan = 0.0036/mV * (35*mV - vm) / (1.0 - exp(-(35*mV - vm) / (12*mV)))/ms : Hz

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + gNa*m**3*h*(ENa-vm) + \
        gl*(El-vm) + gK*n**4*(EK-vm) : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dvm/dt = membrane_Im/C : volt
    """

    neuron = b2.NeuronGroup(1, eqs, method="rk4", dt=dt)
    neuron.vm = -63 * b2.mV
    neuron.I_e = i_ext
    neuron.h = "alphah / (alphah + betah)"
    neuron.n = "alphan / (alphan + betan)"

    st_mon = b2.StateMonitor(neuron, "vm", record=True)
    net = b2.Network(neuron, st_mon)
    net.run(simulation_time)
    return st_mon


sm_pre = simulate_pre_OLM_neuron(1.5 * b2.uA, 200 * b2.ms)

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(sm_pre.t / b2.ms, sm_pre.vm[0] / b2.mV, lw=2, c="k")
ax.set_xlim(0, 200)
ax.set_xlabel("time [ms]")
ax.set_ylabel("v [mV]")
plt.tight_layout()
plt.show()

### The OLM Cell (with h-current, optionally with A-current too)

Adds an h-current (`r`) and, optionally, an A-current (`a`, `b`) to the
pre-OLM model above.

In [ ]:
def simulate_OLM_neuron(i_ext, simulation_time, g_h=12 * b2.msiemens, g_A=0 * b2.msiemens,
                         dt=0.01 * b2.ms):
    El = -70 * b2.mV
    EK = -100 * b2.mV
    ENa = 90 * b2.mV
    EH = -32.9 * b2.mV
    EA = -90 * b2.mV
    gl = 0.05 * b2.msiemens
    gK = 23 * b2.msiemens
    gNa = 30 * b2.msiemens
    C = 1.3 * b2.ufarad

    eqs = """
    I_e : amp

    alpham = (vm + 38*mV) / (10*mV) / (1.0 - exp(-(vm + 38*mV) / (10*mV))) /ms : Hz
    alphah = 0.07 * exp(-(vm + 63.0*mV) / (20.0*mV))/ms : Hz
    alphan = 0.018/mV * (vm - 25*mV) / (1.0 - exp(-(vm - 25*mV) / (25*mV)))/ms : Hz

    betam = 4.0 * exp(-(vm + 65.0*mV) / (18.0*mV))/ms : Hz
    betah = 1.0 / (exp(-(vm + 33.0*mV) / (10.0*mV)) + 1.0)/ms : Hz
    betan = 0.0036/mV * (35*mV - vm) / (1.0 - exp(-(35*mV - vm) / (12*mV)))/ms : Hz

    r_inf = 1.0 / (1.0 + exp((vm + 84.0*mV) / (10.2*mV))) : 1
    tau_r = 1/(exp(-14.59 - 0.086*vm/mV) + exp(-1.87 + 0.0701*vm/mV))*ms : second

    a_inf = 1.0 / (1.0 + exp(-(vm + 14.0*mV) / (16.6*mV))) : 1
    b_inf = 1.0 / (1.0 + exp((vm + 71.0*mV) / (7.3*mV))) : 1
    tau_a = 5*ms : second
    tau_b = 1/(0.000009/exp((vm/mV - 26)/28.5) + 0.014/(0.2 + exp(-(vm/mV + 70.0)/11.0)))*ms : second

    m = alpham / (alpham + betam) : 1
    membrane_Im = I_e + gNa*m**3*h*(ENa-vm) + gl*(El-vm) + gK*n**4*(EK-vm) \
        + g_h*r*(EH-vm) + g_A*a*b*(EA-vm) : amp

    dn/dt = alphan*(1-n)-betan*n : 1
    dh/dt = alphah*(1-h)-betah*h : 1
    dr/dt = (r_inf - r) / tau_r : 1
    da/dt = (a_inf - a) / tau_a : 1
    db/dt = (b_inf - b) / tau_b : 1
    dvm/dt = membrane_Im/C : volt
    """

    neuron = b2.NeuronGroup(1, eqs, method="rk4", dt=dt,
                             namespace={"g_h": g_h, "g_A": g_A})
    neuron.vm = -63 * b2.mV
    neuron.I_e = i_ext
    neuron.h = "alphah / (alphah + betah)"
    neuron.n = "alphan / (alphan + betan)"
    neuron.r = "r_inf"
    neuron.a = "a_inf"
    neuron.b = "b_inf"

    st_mon = b2.StateMonitor(neuron, ["vm", "r", "a", "b"], record=True)
    net = b2.Network(neuron, st_mon)
    net.run(simulation_time)
    return st_mon

### Figure 34.5 (OLM With h-Current)

In [ ]:
sm_h = simulate_OLM_neuron(0 * b2.uA, 200 * b2.ms, g_h=12 * b2.msiemens, g_A=0 * b2.msiemens)

fig, ax = plt.subplots(2, figsize=(7, 4), sharex=True)
ax[0].plot(sm_h.t / b2.ms, sm_h.vm[0] / b2.mV, lw=2, c="k")
ax[0].set_ylabel("v [mV]")
ax[1].plot(sm_h.t / b2.ms, sm_h.r[0], lw=2, c="k")
ax[1].set_ylim(0, 0.01)
ax[1].set_xlabel("time [ms]")
ax[1].set_ylabel("r")
plt.tight_layout()
plt.show()

### Figure 34.8 (OLM With h- and A-Currents)

In [ ]:
sm_ha = simulate_OLM_neuron(0 * b2.uA, 500 * b2.ms, g_h=12 * b2.msiemens, g_A=22 * b2.msiemens)

fig, ax = plt.subplots(2, figsize=(7, 4), sharex=True)
ax[0].plot(sm_ha.t / b2.ms, sm_ha.vm[0] / b2.mV, lw=2, c="k")
ax[0].set_ylabel("v [mV]")
ax[1].plot(sm_ha.t / b2.ms, sm_ha.a[0] * sm_ha.b[0], lw=2, c="k")
ax[1].set_ylim(0, 0.05)
ax[1].set_xlabel("time [ms]")
ax[1].set_ylabel("ab")
plt.tight_layout()
plt.show()